In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Load Database

In [2]:
from pathlib import Path
DB_PATH = Path.cwd().parent / "database" / "car_sales.parquet"

# Read database
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)
df = pd.read_parquet(DB_PATH)
df.head()

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin
0,C_CND_000001,2022-01-02,2,1,2022,Geraldine,Male,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,DoubleÂ Overhead Camshaft,Auto,Black,06457-3834,SUV,8264678,Middletown,"467,740,000.00","242,865,000.00",1,0.05,"467,740,000.00","23,387,000.00","444,353,000.00","347,588,502.33","347,588,502.33","96,764,497.67",21.78
1,C_CND_000002,2022-01-02,2,1,2022,Gia,Male,C & M Motors Inc,Dodge,Durango,DoubleÂ Overhead Camshaft,Auto,Black,60504-7114,SUV,6848189,Aurora,"341,810,000.00","26,625,200,000.00",5,0.10,"1,709,050,000.00","170,905,000.00","1,538,145,000.00","281,435,978.53","1,407,179,892.65","130,965,107.35",8.51
2,C_CND_000003,2022-01-02,2,1,2022,Gianna,Male,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,Red,38701-8047,Passenger,7298798,Greenville,"566,685,000.00","18,619,650,000.00",2,0.02,"1,133,370,000.00","22,667,400.00","1,110,702,600.00","433,036,322.96","866,072,645.92","244,629,954.08",22.02
3,C_CND_000004,2022-01-02,2,1,2022,Giselle,Male,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,Pale White,99301-3882,SUV,6257557,Pasco,"251,860,000.00","242,865,000.00",1,0.50,"251,860,000.00","125,930,000.00","125,930,000.00","190,288,699.80","190,288,699.80","-64,358,699.80",-51.11
4,C_CND_000005,2022-01-02,2,1,2022,Grace,Male,Chrysler Plymouth,Acura,TL,DoubleÂ Overhead Camshaft,Auto,Red,53546-9427,Hatchback,7081483,Janesville,"440,755,000.00","26,355,350,000.00",4,0.10,"1,763,020,000.00","176,302,000.00","1,586,718,000.00","371,011,737.64","1,484,046,950.56","102,671,049.44",6.47


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23905 entries, 0 to 23904
Data columns (total 28 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   car_id           23905 non-null  object 
 1   date             23905 non-null  object 
 2   day              23905 non-null  int32  
 3   month            23905 non-null  int32  
 4   year             23905 non-null  int32  
 5   customer_name    23905 non-null  object 
 6   gender           23905 non-null  object 
 7   dealer_name      23905 non-null  object 
 8   company          23905 non-null  object 
 9   model            23905 non-null  object 
 10  engine           23905 non-null  object 
 11  transmission     23905 non-null  object 
 12  color            23905 non-null  object 
 13  dealer_no_       23905 non-null  object 
 14  body_style       23905 non-null  object 
 15  phone            23905 non-null  int64  
 16  dealer_region    23905 non-null  object 
 17  price       

## Feature engineering

In [4]:
# Create column feature day_of_week, is_weekend, is_workday
df['date'] = pd.to_datetime(df['date'])
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
df['is_workday'] = df['day_of_week'].apply(lambda x: 1 if x < 5 else 0)

In [5]:
# Build Season feature to define the season of the year based on the month
def season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"
    
df["season"] = df["month"].apply(season)

In [6]:
# Adjust date into dividen specific periods
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

In [7]:
# Build feature price_band as arrangement of price into 4 bands
df["price_band"] = pd.qcut(
    df["price"], q=5, labels=[
        "Budget","Economy", "Mid", "Premium", "Luxury"
    ]
)

In [8]:
# Create a feature of discount_level
df['discount_level'] = pd.cut(
    df['discount'], bins=[0,0.05,0.1,0.15,1],
    labels=[
        "Low",
        "Medium",
        "High",
        "Extreme"
    ]
)

# Create feature of weekend only discount by multiplying is_weekend and discount
df['weekend_discount'] = (df['is_weekend'] * df['discount'])

## Data clean

In [9]:
df.head(10)

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin,day_of_week,is_weekend,is_workday,season,quarter,week_of_year,price_band,discount_level,weekend_discount
0,C_CND_000001,2022-01-02,2,1,2022,Geraldine,Male,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,DoubleÂ Overhead Camshaft,Auto,Black,06457-3834,SUV,8264678,Middletown,"467,740,000.00","242,865,000.00",1,0.05,"467,740,000.00","23,387,000.00","444,353,000.00","347,588,502.33","347,588,502.33","96,764,497.67",21.78,6,1,0,Winter,1,52,Mid,Low,0.05
1,C_CND_000002,2022-01-02,2,1,2022,Gia,Male,C & M Motors Inc,Dodge,Durango,DoubleÂ Overhead Camshaft,Auto,Black,60504-7114,SUV,6848189,Aurora,"341,810,000.00","26,625,200,000.00",5,0.10,"1,709,050,000.00","170,905,000.00","1,538,145,000.00","281,435,978.53","1,407,179,892.65","130,965,107.35",8.51,6,1,0,Winter,1,52,Economy,Medium,0.10
2,C_CND_000003,2022-01-02,2,1,2022,Gianna,Male,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,Red,38701-8047,Passenger,7298798,Greenville,"566,685,000.00","18,619,650,000.00",2,0.02,"1,133,370,000.00","22,667,400.00","1,110,702,600.00","433,036,322.96","866,072,645.92","244,629,954.08",22.02,6,1,0,Winter,1,52,Premium,Low,0.02
3,C_CND_000004,2022-01-02,2,1,2022,Giselle,Male,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,Pale White,99301-3882,SUV,6257557,Pasco,"251,860,000.00","242,865,000.00",1,0.50,"251,860,000.00","125,930,000.00","125,930,000.00","190,288,699.80","190,288,699.80","-64,358,699.80",-51.11,6,1,0,Winter,1,52,Budget,Extreme,0.50
4,C_CND_000005,2022-01-02,2,1,2022,Grace,Male,Chrysler Plymouth,Acura,TL,DoubleÂ Overhead Camshaft,Auto,Red,53546-9427,Hatchback,7081483,Janesville,"440,755,000.00","26,355,350,000.00",4,0.10,"1,763,020,000.00","176,302,000.00","1,586,718,000.00","371,011,737.64","1,484,046,950.56","102,671,049.44",6.47,6,1,0,Winter,1,52,Mid,Medium,0.10
5,C_CND_000006,2022-01-02,2,1,2022,Guadalupe,Male,Classic Chevy,Mitsubishi,Diamante,Overhead Camshaft,Manual,Pale White,85257-3102,Hatchback,7315216,Scottsdale,"215,880,000.00","15,291,500,000.00",5,0.02,"1,079,400,000.00","21,588,000.00","1,057,812,000.00","151,544,516.40","757,722,582.00","300,089,418.00",28.37,6,1,0,Winter,1,52,Budget,Low,0.02
6,C_CND_000007,2022-01-02,2,1,2022,Hailey,Male,Clay Johnson Auto Sales,Toyota,Corolla,Overhead Camshaft,Manual,Pale White,78758-7841,Passenger,7727879,Austin,"251,860,000.00","28,784,000,000.00",2,0.02,"503,720,000.00","10,074,400.00","493,645,600.00","196,134,723.96","392,269,447.92","101,376,152.08",20.54,6,1,0,Winter,1,52,Budget,Low,0.02
7,C_CND_000008,2022-01-02,2,1,2022,Graham,Male,U-Haul CO,Mitsubishi,Galant,DoubleÂ Overhead Camshaft,Auto,Pale White,78758-7841,Passenger,6206512,Austin,"755,580,000.00","242,865,000.00",1,0.02,"755,580,000.00","15,111,600.00","740,468,400.00","616,750,863.11","616,750,863.11","123,717,536.89",16.71,6,1,0,Winter,1,52,Luxury,Low,0.02
8,C_CND_000009,2022-01-02,2,1,2022,Naomi,Male,Rabun Used Car Sales,Chevrolet,Malibu,Overhead Camshaft,Manual,Pale White,85257-3102,Hardtop,7194857,Pasco,"1,475,180,000.00","14,661,850,000.00",5,0.05,"7,375,900,000.00","368,795,000.00","7,007,105,000.00","1,195,523,271.39","5,977,616,356.95","1,029,488,643.05",14.69,6,1,0,Winter,1,52,Luxury,Low,0.05
9,C_CND_000010,2022-01-02,2,1,2022,Grayson,Female,Rabun Used Car Sales,Ford,Escort,DoubleÂ Overhead Camshaft,Auto,Pale White,85257-3102,Passenger,7836892,Scottsdale,"269,850,000.00","242,865,000.00",2,0.05,"539,700,000.00","26,985,000.00","512,715,000.00","203,773,492.26","407,546,984.52","105,168,015.48",20.51,6,1,0,Winter,1,52,Budget,Low,0.05


## Data encoding

In [10]:
from sklearn.preprocessing import LabelEncoder

df_ml = df.copy()
le = LabelEncoder()

# Encode the object data in dataframe
for col in df_ml.select_dtypes(include=['object', 'category']).columns:
    df_ml[col] = le.fit_transform(df_ml[col])

In [11]:
df_ml.head()

,car_id,date,day,month,year,customer_name,gender,dealer_name,company,model,engine,transmission,color,dealer_no_,body_style,phone,dealer_region,price,income_customer,quantity,discount,gross_sales,discount_amount,sales,cost,total_cost,profit,profit_margin,day_of_week,is_weekend,is_workday,season,quarter,week_of_year,price_band,discount_level,weekend_discount
0,0,2022-01-02,2,1,2022,1050,1,0,8,60,0,0,0,0,3,8264678,4,"467,740,000.00","242,865,000.00",1,0.05,"467,740,000.00","23,387,000.00","444,353,000.00","347,588,502.33","347,588,502.33","96,764,497.67",21.78,6,1,0,3,1,52,3,1,0.05
1,1,2022-01-02,2,1,2022,1057,1,1,7,52,0,0,0,3,3,6848189,0,"341,810,000.00","26,625,200,000.00",5,0.10,"1,709,050,000.00","170,905,000.00","1,538,145,000.00","281,435,978.53","1,407,179,892.65","130,965,107.35",8.51,6,1,0,3,1,52,1,2,0.10
2,2,2022-01-02,2,1,2022,1061,1,2,4,57,1,1,2,1,2,7298798,2,"566,685,000.00","18,619,650,000.00",2,0.02,"1,133,370,000.00","22,667,400.00","1,110,702,600.00","433,036,322.96","866,072,645.92","244,629,954.08",22.02,6,1,0,3,1,52,4,1,0.02
3,3,2022-01-02,2,1,2022,1068,1,4,27,36,1,1,1,6,3,6257557,5,"251,860,000.00","242,865,000.00",1,0.50,"251,860,000.00","125,930,000.00","125,930,000.00","190,288,699.80","190,288,699.80","-64,358,699.80",-51.11,6,1,0,3,1,52,0,0,0.50
4,4,2022-01-02,2,1,2022,1083,1,3,0,141,0,0,2,2,1,7081483,3,"440,755,000.00","26,355,350,000.00",4,0.10,"1,763,020,000.00","176,302,000.00","1,586,718,000.00","371,011,737.64","1,484,046,950.56","102,671,049.44",6.47,6,1,0,3,1,52,3,2,0.10


## Split data into X and y

In [12]:
from sklearn.model_selection import train_test_split

X = df_ml[["gender","income_customer", "dealer_name", "dealer_region", "company", "model", "engine", "transmission",
    "color", "body_style", "price", "discount", "day_of_week", "season", "quarter", "week_of_year"]]
y = df_ml["quantity"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")

X_train shape: (19124, 16), y_train shape: (19124,)


In [13]:
X_train.head(2)

,gender,income_customer,dealer_name,dealer_region,company,model,engine,transmission,color,body_style,price,discount,day_of_week,season,quarter,week_of_year
19155,1,"29,521,590,000.00",23,2,17,98,0,0,1,0,"379,589,000.00",0.02,6,0,4,39
10018,1,"32,112,150,000.00",12,6,28,106,0,0,0,1,"170,905,000.00",0.02,6,3,4,50


### Machine learning models - Demand prediction (Quantity as a target)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

models = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0),
    "XGBoost": XGBRegressor(random_state=42, verbosity=0, objective='reg:squarederror', tree_method='hist')
}

parameters = {
    "Decision Tree": {
        "max_depth": [None, 5, 10, 15],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4, 8]
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 15, 20, None],
        "min_samples_split": [2, 5, 10, 20],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"]
    },
    "CatBoost": {
        "iterations": [200, 300],
        "depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5, 7]
    },
    "XGBoost": {
        "n_estimators": [200, 300],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.8, 0.9, 1.0],
        "colsample_bytree": [0.8, 0.9, 1.0]
    }
}

# Train and evaluate models
trained_models = {}
results = []
feature_importances = {}

for model_name, model in models.items():
    print("=" * 50)
    print(f"Training {model_name}...")
    print("=" * 50)

    # Machine learning models with hyperparameter tuning using RandomizedSearchCV
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=parameters[model_name],
        n_iter=10,
        cv=5,
        scoring='r2',
        n_jobs=-1,
        verbose=1,
        random_state=42
    )
    random_search.fit(X_train, y_train) # Train the model

    # Get the best model from RandomizedSearchCV
    best_model = random_search.best_estimator_
    print(f"Best parameters for {model_name}: {random_search.best_params_}")
    
    # Store the trained model
    trained_models[model_name] = best_model

    # Best parameters
    print("\nBest Parameters")
    print(random_search.best_params_)

    # Predict on the test set
    y_pred = best_model.predict(X_test)

    # Evaluate the model
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Append results to the list
    results.append({
        "Model": model_name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    })

    print(f"{model_name} Evaluation Metrics:")
    print(f"Mean Squared Error (MSE): {mse:.3f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
    print(f"Mean Absolute Error (MAE): {mae:.3f}")
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.3f}")
    print(f"R-squared (R2): {r2:.3f}")

    # Feature importance
    if hasattr(best_model, 'feature_importances_'):
        importance = pd.DataFrame({
            "Feature": X_train.columns,
            "Importance": best_model.feature_importances_
        }).sort_values(by="Importance", ascending=False)
        feature_importances[model_name] = importance

# ==========================================================
# Comparison Table
# ==========================================================
df_comparison = pd.DataFrame(results).sort_values(by="R2", ascending=False).reset_index(drop=True)
print("\nModel Comparison:")
print(df_comparison)

# Feature importance for each model
for model_name, importance in feature_importances.items():
    print("\n")
    print("=" * 50)
    print(f"{model_name} Feature Importance:")
    print("=" * 50)
    print(importance.head(20))

Training Decision Tree...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters for Decision Tree: {'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 5}

Best Parameters
{'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 5}
Decision Tree Evaluation Metrics:
Mean Squared Error (MSE): 1.143
Root Mean Squared Error (RMSE): 1.069
Mean Absolute Error (MAE): 0.864
Mean Absolute Percentage Error (MAPE): 0.497
R-squared (R2): 0.335


AttributeError: module 'pandas' has no attribute 'DataFraame'

In [ ]:
importance = (
    pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": trained_models["Random Forest"].feature_importances_
    })
    .sort_values("Importance", ascending=False)
)

print(importance)

            Feature  Importance
10  income_customer        0.33
3             model        0.13
2           company        0.08
16     week_of_year        0.08
1       dealer_name        0.07
6             color        0.07
11      day_of_week        0.04
9     dealer_region        0.04
8        body_style        0.04
7        dealer_no_        0.04
5      transmission        0.02
14           season        0.02
4            engine        0.02
0            gender        0.01
15          quarter        0.01
12       is_weekend        0.00
13       is_workday        0.00


## Save models